# Estimativa de pose em vídeo com ViTPose

Lê um vídeo, extrai 17 pontos do corpo em cada quadro, desenha os pontos sobre os quadros e calcula mapas de calor.

Antes de rodar: `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)`.

## 1. Ambiente

In [ ]:
!pip install -q "transformers>=4.49"

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.ndimage import gaussian_filter
from scipy.signal import savgol_filter
from tqdm.auto import tqdm
from transformers import AutoProcessor, RTDetrForObjectDetection, VitPoseForPoseEstimation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

## 2. Vídeo

In [ ]:
VIDEO = ""           # caminho do vídeo; vazio = enviar um arquivo
MAX_SEGUNDOS = 20    # quanto do vídeo processar
PASSO = 1            # 1 = todos os quadros; 2 = um a cada dois

if not VIDEO:
    from google.colab import files
    VIDEO = next(iter(files.upload()))

In [ ]:
cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS)
N_QUADROS = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
LARGURA = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
ALTURA = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"{LARGURA}x{ALTURA}, {FPS:.2f} fps, {N_QUADROS} quadros, {N_QUADROS / FPS:.1f} s")

## 3. Modelos

Dois modelos em sequência: o RT-DETR acha as pessoas no quadro e o ViTPose localiza os 17 pontos dentro de cada caixa.

In [ ]:
#Detector de pessoas: RT-DETR identifica as pessoas presentes na imagem
#e delimita suas regiões por meio de caixas delimitadoras (bounding boxes).
DETECTOR = "PekingU/rtdetr_r50vd_coco_o365"
# ViTPose: estima a localização dos pontos-chave do corpo humano
# dentro das regiões detectadas, como ombros, cotovelos, quadris e joelhos.
POSE = "usyd-community/vitpose-base-simple"
#nível mínimo de confiança utilizado para aceitar uma detecção de pessoa
LIMIAR_DETECCAO = 0.3

#Carrega os componentes responsáveis por preparar as imagens para cada modelo e organizar suas entradas.
det_proc = AutoProcessor.from_pretrained(DETECTOR)
det_model = RTDetrForObjectDetection.from_pretrained(DETECTOR).to(DEVICE).eval()
pose_proc = AutoProcessor.from_pretrained(POSE)

pose_model = VitPoseForPoseEstimation.from_pretrained(POSE).to(DEVICE).eval()

In [ ]:
# Lista os 17 pontos-chave retornados pelo modelo
PONTOS = [
    "Nariz", "Olho_Esquerdo", "Olho_Direito", "Orelha_Esquerda", "Orelha_Direita",
    "Ombro_Esquerdo", "Ombro_Direito", "Cotovelo_Esquerdo", "Cotovelo_Direito",
    "Pulso_Esquerdo", "Pulso_Direito", "Anca_Esquerda", "Anca_Direita",
    "Joelho_Esquerdo", "Joelho_Direito", "Tornozelo_Esquerdo", "Tornozelo_Direito",
]
# Define as conexões (esqueleto) entre os pontos por meio de pares de índices
ARESTAS = [
    (15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12), (5, 6), (5, 7), (6, 8),
    (7, 9), (8, 10), (1, 2), (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6),
]
# Identifica cada ponto como pertencente ao lado esquerdo, direito ou ao centro do corpo
LADO = ["centro"] + ["esquerdo", "direito"] * 8       # lado do corpo de cada ponto
# Associa cores distintas aos lados
COR = {"esquerdo": "#2a78d6", "direito": "#eb6834", "centro": "#eda100"}
# Confiança mínima para usar um ponto
LIMIAR_PONTO = 0.3

## 4. Extração dos pontos

In [ ]:
@torch.no_grad()
def pose_do_quadro(quadro_bgr):
  # Pontos de todas as pessoas do quadro: array (pessoas, 17, 3) com x, y e score.
	# Converte o quadro para RGB
  img = Image.fromarray(cv2.cvtColor(quadro_bgr, cv2.COLOR_BGR2RGB))
	# Faz a detecção de pessoas usando o RT-DETR
  entrada = det_proc(images=img, return_tensors="pt").to(DEVICE)
  det = det_proc.post_process_object_detection(
    det_model(**entrada), target_sizes=torch.tensor([(img.height, img.width)]), threshold=LIMIAR_DETECCAO)[0]

	# Seleciona apenas a classe 0 -> pessoas
  caixas = det["boxes"][det["labels"] == 0].cpu().numpy()
	# Se nenhuma pessoa for detectada, retorna um array vazio
  if len(caixas) == 0:
    return np.empty((0, 17, 3))
	# Converte as coordenadas (x1, y1, x2, y2) -> (x, y, largura, altura)
  caixas[:, 2:] -= caixas[:, :2]

	# Estimar os pontos-chave de cada pessoa
  entrada = pose_proc(img, boxes=[caixas], return_tensors="pt").to(DEVICE)
  pessoas = pose_proc.post_process_pose_estimation(pose_model(**entrada), boxes=[caixas])[0]
	# Array (pessoas, 17, 3): 17 pontos-chave -> coordenadas x, y e pelo score de confiança.
  return np.stack([np.column_stack([p["keypoints"].cpu().numpy(), p["scores"].cpu().numpy()]) for p in pessoas])

Com mais de uma pessoa em cena: no primeiro quadro fica a de maior score médio; nos seguintes, a mais próxima da escolhida no quadro anterior.

In [ ]:
# Define quantos quadros serão considerados
cap = cv2.VideoCapture(VIDEO)
limite = min(N_QUADROS, int(MAX_SEGUNDOS * FPS))
quadros, kpts, centro = [], [], None
for i in tqdm(range(limite)): # Processa cada quadro
    ok, quadro = cap.read()
    if not ok:
        break
    if i % PASSO:
        continue
    # Detecta as pessoas e estima seus pontos-chave
    pessoas = pose_do_quadro(quadro)
    if len(pessoas):
        # Múltiplas pessoas? Seleciona a de maior confiança média
        centros = pessoas[:, :, :2].mean(axis=1)
        if centro is None:
            j = pessoas[:, :, 2].mean(axis=1).argmax()
        else:
            j = np.linalg.norm(centros - centro, axis=1).argmin()
        centro = centros[j]
        kpts.append(pessoas[j])
    else:# Ausência de detecção: preenche com NaN
        kpts.append(np.full((17, 3), np.nan))
    quadros.append(i)
cap.release()
# Armazena os resultados
quadros = np.array(quadros)
kpts = np.stack(kpts)# (quadros, 17, 3)
tempo = quadros / FPS
print(kpts.shape)

In [ ]:
# Cria uma tabela (DataFrame) para organiza os dados em colunas
tabela = pd.DataFrame({
    "quadro": np.repeat(quadros, 17),
    "tempo_s": np.repeat(tempo, 17),
    "ponto": np.tile(PONTOS, len(quadros)),
    "x": kpts[:, :, 0].ravel(),
    "y": kpts[:, :, 1].ravel(),
    "score": kpts[:, :, 2].ravel(),
})
# Salva os resultados num arquivo CSV
tabela.to_csv("pontos.csv", index=False)
tabela.head(17)

## 5. Pontos sobre os quadros

In [ ]:
def ler_quadro(n):
    #Lê um quadro específico do vídeo (n)
    cap = cv2.VideoCapture(VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n))
    ok, quadro = cap.read()
    cap.release()
    return cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)

def desenhar(ax, k):
    # Desenha o esqueleto de um quadro
    # Considera apenas os pontos com determinada confiança
    visivel = k[:, 2] >= LIMIAR_PONTO
    for a, b in ARESTAS:
        # Usa as informações de COR e ARESTAS definidas antes
        if visivel[a] and visivel[b]:
            cor = COR[LADO[a]] if LADO[a] == LADO[b] else "white"
            ax.plot(k[[a, b], 0], k[[a, b], 1], color=cor, lw=2)
    for lado, cor in COR.items():
        m = visivel & (np.array(LADO) == lado)
        # Desenha as articulações como marcadores coloridos
        ax.scatter(k[m, 0], k[m, 1], s=22, color=cor, edgecolor="white", linewidth=0.6, zorder=3, label=lado)

In [ ]:
# Seleciona 6 frames distribuídos ao longo do vídeo
escolhidos = np.linspace(0, len(quadros) - 1, 6).astype(int)
# Organiza a visualização em uma grade 2 x 3
fig, eixos = plt.subplots(2, 3, figsize=(15, 10 * ALTURA / LARGURA))
for ax, j in zip(eixos.ravel(), escolhidos):
    # Usa as funções anteriores para ler e desenhar um frame
    ax.imshow(ler_quadro(quadros[j]))
    desenhar(ax, kpts[j])
    ax.set_title(f"quadro {quadros[j]} ({tempo[j]:.1f} s)")
    ax.axis("off")
eixos[0, 0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 6. Vídeo com o esqueleto

In [ ]:
def bgr(cor_hex):
    # Converte as cores: hexadecimal -> BGR (OpenCV)
    r, g, b = (int(cor_hex[i:i + 2], 16) for i in (1, 3, 5))
    return (b, g, r)

cap = cv2.VideoCapture(VIDEO)
saida = cv2.VideoWriter("pose_tmp.mp4", cv2.VideoWriter_fourcc(*"mp4v"), FPS / PASSO, (LARGURA, ALTURA))
# Ajusta a largura das linhas e o tamanho dos pontos conforme a resolução do vídeo
espessura = max(2, LARGURA // 400)
for j, n in enumerate(tqdm(quadros)): # Percorre os quadros selecionados
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n))
    ok, quadro = cap.read()
    k = kpts[j]
    # Seleciona os pontos-chave e desenha os pontos e as conexões entre as articulações
    visivel = k[:, 2] >= LIMIAR_PONTO
    for a, b in ARESTAS:
        if visivel[a] and visivel[b]:
            cor = bgr(COR[LADO[a]]) if LADO[a] == LADO[b] else (255, 255, 255)
            cv2.line(quadro, tuple(k[a, :2].astype(int)), tuple(k[b, :2].astype(int)), cor, espessura)
    for p in np.flatnonzero(visivel):
        cv2.circle(quadro, tuple(k[p, :2].astype(int)), espessura * 2, bgr(COR[LADO[p]]), -1)
    saida.write(quadro)
cap.release()
saida.release()

In [ ]:
!ffmpeg -y -loglevel error -i pose_tmp.mp4 -vcodec libx264 -pix_fmt yuv420p pose.mp4

from IPython.display import Video
Video("pose.mp4", embed=True, width=640)

## 7. Trajetórias

In [ ]:
# Seleciona os pontos que queremos visualizar
EXTREMIDADES = {"Pulso_Esquerdo": "#2a78d6", "Pulso_Direito": "#eb6834",
                "Tornozelo_Esquerdo": "#1baf7a", "Tornozelo_Direito": "#4a3aa7"}

fig, ax = plt.subplots(figsize=(10, 10 * ALTURA / LARGURA))
# Seleciona o primerio frame como referência
ax.imshow(ler_quadro(quadros[0]), alpha=0.6)
for nome, cor in EXTREMIDADES.items():
    # Recupera as coordenadas (x, y) dos pontos (com confiança)
    k = kpts[:, PONTOS.index(nome)]
    xy = np.where(k[:, 2:] >= LIMIAR_PONTO, k[:, :2], np.nan)
    # Exibe sobre a imagem
    ax.plot(xy[:, 0], xy[:, 1], color=cor, lw=1.2, label=nome)
ax.legend(loc="upper left", fontsize=8)
ax.axis("off")
plt.show()

## 8. Mapas de calor

Onde os pontos passaram mais tempo (`presenca`) e onde houve mais deslocamento (`movimento`).

In [ ]:
def mapa_de_calor(indices, tipo="presenca", sigma=None):
    # Mapa (ALTURA x LARGURA), de 0 a 1, dos pontos dados.
    # presenca:  cada observação do ponto soma 1 na posição em que ele está.
    # movimento: cada observação soma o deslocamento desde o quadro anterior.
    sigma = sigma or 0.02 * max(LARGURA, ALTURA)
    mapa = np.zeros((ALTURA, LARGURA), dtype=np.float32)
    for i in indices:
        xy = np.where(kpts[:, i, 2:] >= LIMIAR_PONTO, kpts[:, i, :2], np.nan)
        if tipo == "movimento":
            peso = np.linalg.norm(np.diff(xy, axis=0), axis=1)
            xy = xy[1:]
        else:
            peso = np.ones(len(xy))
        ok = ~np.isnan(xy).any(axis=1) & ~np.isnan(peso)
        x = np.clip(xy[ok, 0].round().astype(int), 0, LARGURA - 1)
        y = np.clip(xy[ok, 1].round().astype(int), 0, ALTURA - 1)
        np.add.at(mapa, (y, x), peso[ok])
    mapa = gaussian_filter(mapa, sigma)# Suavização
    return mapa / mapa.max() if mapa.max() > 0 else mapa

def mostrar_mapa(ax, mapa, titulo): #Exibe o mapa de calor
    ax.imshow(cv2.cvtColor(ler_quadro(quadros[0]), cv2.COLOR_RGB2GRAY), cmap="gray", vmin=0, vmax=255)
    im = ax.imshow(mapa, cmap="inferno", alpha=np.clip(mapa * 3, 0, 0.85), vmin=0, vmax=1)
    ax.set_title(titulo)
    ax.axis("off")
    return im

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(15, 7.5 * ALTURA / LARGURA))
mostrar_mapa(eixos[0], mapa_de_calor(range(17), "presenca"), "presença, 17 pontos")
im = mostrar_mapa(eixos[1], mapa_de_calor(range(17), "movimento"), "movimento, 17 pontos")
fig.colorbar(im, ax=eixos, fraction=0.02, label="intensidade relativa")
plt.show()

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(13, 13 * ALTURA / LARGURA))
for ax, nome in zip(eixos.ravel(), EXTREMIDADES):
    mostrar_mapa(ax, mapa_de_calor([PONTOS.index(nome)], "movimento"), nome)
plt.tight_layout()
plt.show()

## 9. Velocidade ao longo do tempo

A trajetória é suavizada antes de derivar, e a velocidade é dividida pelo comprimento do tronco para não depender da distância da câmera.

In [ ]:
def trajetoria(i):
    # Obtém as coordenadas (x, y) de um ponto-chave em todos os quadros do vídeo
    # Filtro de Savitzky-Golay: reduz oscilações na trajetória
    xy = np.where(kpts[:, i, 2:] >= LIMIAR_PONTO, kpts[:, i, :2], np.nan)
    xy = pd.DataFrame(xy).interpolate(limit_direction="both").values
    return savgol_filter(xy, 11, 3, axis=0)

# Calcula pontos médios entre os ombros e entre as ancas (quadris)
ombros = (trajetoria(5) + trajetoria(6)) / 2
ancas = (trajetoria(11) + trajetoria(12)) / 2
# Estima o comprimento do tronco pela mediana da distância
tronco = np.median(np.linalg.norm(ombros - ancas, axis=1))
print(f"tronco: {tronco:.0f} px")

fig, ax = plt.subplots(figsize=(12, 4))
for nome, cor in EXTREMIDADES.items():
    # Calcula a velocidade a partir do deslocamento entre quadros consecutivos. normalizada pelo tronco
    xy = trajetoria(PONTOS.index(nome))
    velocidade = np.linalg.norm(np.diff(xy, axis=0), axis=1) / np.diff(tempo) / tronco
    ax.plot(tempo[1:], velocidade, color=cor, lw=1.2, label=nome)
ax.set_xlabel("tempo (s)")
ax.set_ylabel("velocidade (troncos/s)")
ax.legend(fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.15), frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()